# Laboratorio L5 - Robustezza delle password

B.4 - Cybersicurezza e IA

Il notebook adotta il punto di vista di chi gestisce un servizio: calcola quanto è difficile indovinare una password e controlla le password proposte dagli utenti al momento della registrazione.

Regole:

- eseguire le celle in ordine, dall'alto in basso (`Maiusc+Invio`)
- non inserire password reali: usare solo esempi inventati

## 1. Entropia di una password casuale

Per una password generata a caso con `L` caratteri scelti da un alfabeto di `N` simboli:

entropia = L × log2(N) bit

- `math.log2(n)`: logaritmo in base 2 di `n`, cioè il numero di bit necessari a rappresentare `n` possibilità
- `round(x, 1)`: arrotonda `x` a una cifra decimale

In [ ]:
import math

ALFABETI = {
    "solo minuscole (26)": 26,
    "minuscole, maiuscole, cifre (62)": 62,
    "tutti i caratteri stampabili (94)": 94,
}

def entropia(lunghezza, n_simboli):
    return lunghezza * math.log2(n_simboli)

for nome, n in ALFABETI.items():
    for lunghezza in [8, 12, 15, 20]:
        print(f"{nome:36} lunghezza {lunghezza:2}: {entropia(lunghezza, n):5.1f} bit")
    print()

### Grafico: entropia in funzione della lunghezza

`matplotlib` disegna una linea per ciascun alfabeto.

In [ ]:
import matplotlib.pyplot as plt

lunghezze = list(range(4, 25))
for nome, n in ALFABETI.items():
    plt.plot(lunghezze, [entropia(l, n) for l in lunghezze], marker="o", markersize=3, label=nome)
plt.xlabel("lunghezza (caratteri)")
plt.ylabel("entropia (bit)")
plt.title("Entropia di password casuali")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**Domanda (esercizio 1)**: conviene aggiungere quattro caratteri a una password di 8 minuscole, o passare da 8 minuscole a 8 caratteri di ogni tipo? Rispondere confrontando i valori.

Risposta:

## 2. Tempo per provare tutte le combinazioni

Una password con entropia di `b` bit ha `2**b` combinazioni possibili (`**` è l'elevamento a potenza).

Le tre velocità di attacco sono ipotesi didattiche, espresse come ordini di grandezza:

- in linea, con limitazione dei tentativi da parte del servizio: 10 tentativi al secondo (in realtà molto meno, perché il servizio blocca l'account dopo pochi errori)
- fuori linea, archivio protetto con una funzione di hash lenta: 10 000 tentativi al secondo
- fuori linea, archivio protetto con una funzione di hash veloce e inadatta alle password: 10 miliardi di tentativi al secondo

In [ ]:
VELOCITA = {
    "in linea, con limitazione": 10,
    "fuori linea, hash lento": 10_000,
    "fuori linea, hash veloce": 10_000_000_000,
}

SECONDI_ANNO = 365.25 * 24 * 3600

def durata_leggibile(secondi):
    if secondi < 60:
        return f"{secondi:.1f} secondi"
    if secondi < 3600:
        return f"{secondi/60:.1f} minuti"
    if secondi < 86400:
        return f"{secondi/3600:.1f} ore"
    if secondi < SECONDI_ANNO:
        return f"{secondi/86400:.1f} giorni"
    return f"{secondi/SECONDI_ANNO:.2e} anni"

for descrizione, bit in [("8 minuscole", entropia(8, 26)),
                         ("8 caratteri di ogni tipo", entropia(8, 94)),
                         ("15 minuscole", entropia(15, 26))]:
    print(f"{descrizione} ({bit:.1f} bit)")
    for nome, v in VELOCITA.items():
        print(f"   {nome:28}: {durata_leggibile(2**bit / v)}")
    print()

**Domanda (esercizio 2)**: perché la stessa password può resistere per secoli in un caso e cadere in poche ore in un altro? Da chi dipende la differenza?

Risposta:

Nota: questi tempi valgono solo per password **generate a caso**. Una password scelta da una persona viene trovata molto prima, perché gli attacchi a dizionario provano per prime le scelte più comuni.

## 3. Il controllo di un servizio al momento della registrazione

La funzione `valuta_password` applica le raccomandazioni NIST SP 800-63B-4:

- lunghezza minima di 15 caratteri (se la password è l'unico fattore)
- rifiuto delle password presenti in una lista di blocco
- rifiuto delle password costruite con schemi prevedibili: una parola della lista seguita da numeri o simboli, eventualmente con sostituzioni come `4` al posto di `a`

Elementi di Python usati:

- `set`: insieme di valori, per verificare velocemente se un valore è presente con `in`
- `str.lower()`: converte in minuscolo
- `str.translate(tabella)`: sostituisce caratteri secondo una tabella creata con `str.maketrans`
- `re.fullmatch(schema, testo)`: verifica se tutto il testo corrisponde a uno schema (espressione regolare). Nello schema usato, `(.+?)` indica la parte iniziale, la più breve possibile, e `[0-9!@#$%&*._-]*` zero o più cifre o simboli fino alla fine: la parte iniziale è quindi la parola che precede i numeri e i simboli finali

In [ ]:
import re

LISTA_DI_BLOCCO = {
    "123456", "123456789", "password", "qwerty", "qwertyuiop", "111111", "abc123",
    "iloveyou", "admin", "letmein", "benvenuto", "ciao", "amore", "juventus", "inter",
    "milan", "roma", "napoli", "lazio", "estate", "inverno", "autunno", "primavera",
    "calcio", "forza", "scuola", "studente", "password1", "passw0rd",
}

# sostituzioni comuni: cifre e simboli usati al posto di lettere
SOSTITUZIONI = str.maketrans({"4": "a", "@": "a", "3": "e", "1": "i", "!": "i", "0": "o", "5": "s", "$": "s", "7": "t"})

def normalizza(parola):
    return parola.lower().translate(SOSTITUZIONI)

def valuta_password(password, lista=LISTA_DI_BLOCCO, minimo=15):
    motivi = []
    if len(password) < minimo:
        motivi.append(f"troppo corta ({len(password)} caratteri, minimo {minimo})")
    if password.lower() in lista or normalizza(password) in lista:
        motivi.append("presente nella lista di blocco")
    # schema prevedibile: parola (anche con sostituzioni) seguita da cifre e simboli finali
    m = re.fullmatch(r"(.+?)[0-9!@#$%&*._-]*", password.lower())
    base = normalizza(m.group(1)) if m else ""
    if base in lista and base != password.lower():
        motivi.append(f"schema prevedibile: parola '{base}' seguita da numeri o simboli")
    return ("RIFIUTATA", motivi) if motivi else ("ACCETTATA", [])

Password di esempio da valutare (inventate, da non usare):

In [ ]:
ESEMPI = [
    "juventus2008",
    "Estate2026!",
    "P4ssw0rd!",
    "Scu0la2026",
    "ilmiocanesichiamaFuffy",
    "tavolo-pianeta-muschio-lanterna",
    "Tr0ub4dor&3",
    "qwertyuiop",
    "k7#Qm2!vLp9@xZ4r",
]

for p in ESEMPI:
    esito, motivi = valuta_password(p)
    print(f"{p:34} {esito}")
    for motivo in motivi:
        print(f"{'':34}   - {motivo}")

**Domande (esercizio 3)**

- Perché `Estate2026!` viene rifiutata, pur contenendo maiuscole, cifre e un simbolo?
- `ilmiocanesichiamaFuffy` viene accettata dal controllo automatico. È davvero una buona password? Che cosa il controllo non può sapere?
- `Tr0ub4dor&3` è un esempio famoso: perché, anche se accettata dalla lunghezza minima di altri siti, qui viene rifiutata?

Risposte:

## 4. Parole legate al contesto (approfondimento)

NIST raccomanda di includere nella lista di blocco le parole legate al servizio: nome del servizio, dell'organizzazione, della città. Aggiungere alla lista alcune parole (per esempio il nome della scuola e della città) e rieseguire la valutazione.

`lista | {...}` crea un nuovo insieme con gli elementi di entrambi.

In [ ]:
PAROLE_CONTESTO = {"galilei", "itis", "romaitalia"}   # modificare con parole del proprio contesto

ALTRI_ESEMPI = ["Galilei2026!!!!!", "itis-galilei-roma", "G4lil3i_laboratorio"]
for p in ALTRI_ESEMPI:
    esito, motivi = valuta_password(p, LISTA_DI_BLOCCO | PAROLE_CONTESTO)
    print(f"{p:24} {esito} {motivi}")

**Domanda**: perché le parole legate al servizio sono tra le prime che un attaccante prova?

Risposta: